# Pipeline

Весь пайплайн генерации перечня целевых показателей

In [1]:
from langchain_ollama import OllamaEmbeddings

embed = OllamaEmbeddings(model='nomic-embed-text:latest', base_url='http://a.dgx:11434/')

In [10]:
from langchain_community.document_loaders import Docx2txtLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

documents = Docx2txtLoader('ССЭР РФ.docx').load()

In [ ]:
text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=100, length_function=len)

split_docs = text_splitter.split_documents(documents)
len(split_docs)

In [5]:
from langchain_core.vectorstores import InMemoryVectorStore

store = InMemoryVectorStore(embed)

In [1]:
from src import Model
from pydantic import Field

class InhabitedLocalityModel(Model):
    """
    Административные сведения о населенном пункте
    """
    name : str = Field(description='Тип и название населенного пункта')

    settlement : str | None = Field(default=None, description='Городское или сельское поселение')
    municipal : str | None = Field(default=None, description='Муниципальный или городской округ, муниципальный район')

    federal_subject : str = Field(description='Субъект Российской Федерации')
    federal_district : str = Field(description='Федеральный округ')

query = InhabitedLocalityModel(
    name='город Гатчина',
    settlement='Бывш. Гатчинское городское поселение',
    municipal='Гатчинский муниципальный округ (бывш. Гатчинский муниципальный район)',
    federal_subject='Ленинградская область',
    federal_district='Северо-Западный федеральный округ'
)

In [2]:
from langgraph.graph import StateGraph
from typing import TypedDict

class State(Model):
    inhabited_locality : InhabitedLocalityModel
    # analysis : AnalysisModel
    # documents : DocumentsModel

In [5]:
from src import Agent, llm, lightrag_retrieve

agent = Agent(llm=llm, tools=[lightrag_retrieve], system_prompt='Тебе необходимо найти информацию в стратегии пространственного развития, связанную с данной территорией', debug=True)

In [ ]:
print(res)